In [ ]:
%pip install admet-ai

In [2]:
import csv
import os
import sys
import config
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, QED, RDConfig
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem import Lipinski
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP

from admet_ai import ADMETModel

### Disable RDKit informational messages ###
RDLogger.DisableLog('rdApp.*')

#############
### Setup ###
#############

### Load SMILES ###
standardized_compounds = config.STANDARDIZED_SMILES
supply = Chem.SmilesMolSupplier(
    standardized_compounds,
    delimiter='\t',
    titleLine=False
    )

### ADMET ###
model = ADMETModel(include_physchem=False)
admet_endpoints = [
            "Solubility_AqSolDB",
            "hERG",
            "Caco2_Wang",
            "BBB_Martins",
            "Clearance_Hepatocyte_AZ",
            "Clearance_Microsome_AZ",
            "PAMPA_NCATS",
            "PPBR_AZ"
            ]

### PAINS, Brenk, NIH ###

# PAINS flag
params_pains = FilterCatalogParams()
params_pains.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
catalog_pains = FilterCatalog(params_pains)

# Brenk Flag
params_unwanted = FilterCatalogParams()
params_unwanted.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
catalog_unwanted = FilterCatalog(params_unwanted)

# NIH Flag
params_nih = FilterCatalogParams()
params_nih.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)
catalog_nih = FilterCatalog(params_nih)

### Synthesizability score ###
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
# now you can import sascore!
import sascorer

###############################
### Filtering and Profiling ###
###############################

def calculate_pains_brenk(mol):

    # Check for PAINS
    pains_match = catalog_pains.HasMatch(mol)

    # Check for BRENK
    brenk_match = catalog_unwanted.HasMatch(mol)

    # Check for NIH
    NIH_match = catalog_nih.HasMatch(mol)
    
    return pains_match, brenk_match, NIH_match


def profile_compound(mol, admet_model=None):

    smiles = Chem.MolToSmiles(mol)

    # -----------------------------------------
    # 1. PAINS / BRENK
    # -----------------------------------------

    pains_match, brenk_match, NIH_match = calculate_pains_brenk(mol)

    # -----------------------------------------
    # 2. RDKit descriptors
    # -----------------------------------------

    profile = {
        "Compound": smiles,
        "MW": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Crippen.MolLogP(mol), 2),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "Aromatic_Ring_Number": rdMolDescriptors.CalcNumAromaticRings(mol),
        "TPSA": round(Descriptors.TPSA(mol), 2),
        "RotB": Descriptors.NumRotatableBonds(mol),
        "QED": round(QED.qed(mol), 2),
        "Formal_Charge": Chem.GetFormalCharge(mol),
        "SA_score": round(sascorer.calculateScore(mol),2),
        "PAINS": pains_match,
        "BRENK": brenk_match,
        "NIH": NIH_match
    }

    # -----------------------------------------
    # 3. ADMET
    # -----------------------------------------

    if admet_model is not None:

        preds = admet_model.predict(smiles=smiles)

        for endpoint in admet_endpoints:

            value = preds.get(endpoint)

            profile[endpoint] = (
                round(value, 2)
                if value is not None
                else None
            )

    else:

        for endpoint in admet_endpoints:
            profile[endpoint] = None

    return profile # Returns a dictionary containing the above defined ADMET parameters

profile_table = []

# Iterate through all compounds in smi file
for i, mol in enumerate(supply, start=0):
    if mol is not None:
        profile_table.append(profile_compound(mol, admet_model=model))
        
##############
### Output ###
##############

### Save results as csv file ###
df = pd.DataFrame(profile_table)
profile_output = config.PROFILE_OUTPUT
df.to_csv(profile_output, index=False)

print(f'The results have been successfully saved as {profile_output}!') 

SMILES to Mol: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1040.00it/s]
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 1, batch_size = 64)
GPU available: False, used: False                                                                | 0/2 [00:00<?, ?it/s]
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does 

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

model ensembles: 100%|███████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  2.52it/s]

The results have been successfully saved as results/compound_profiles.csv!


In [6]:
# predse = model.predict(smiles='CC1(NC(=O)[C@H](F)Oc2cccnc2)CN(S(N)(=O)=O)C1')
# predse

SMILES to Mol: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 860.72it/s]
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 1, batch_size = 64)
GPU available: False, used: False                                                                | 0/2 [00:00<?, ?it/s]
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\pham.congdat\AppData\Local\anaconda3\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Output()

Output()

...

Output()

Output()

Output()

model ensembles: 100%|███████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  2.52it/s]


{'AMES': 0.49017781019210815,
 'BBB_Martins': 0.7113314270973206,
 'Bioavailability_Ma': 0.8457330465316772,
 'CYP1A2_Veith': 0.003848641412332654,
 'CYP2C19_Veith': 0.06193540617823601,
 'CYP2C9_Substrate_CarbonMangels': 0.1525578796863556,
 'CYP2C9_Veith': 0.02486288920044899,
 'CYP2D6_Substrate_CarbonMangels': 0.05251621454954147,
 'CYP2D6_Veith': 0.011781260371208191,
 'CYP3A4_Substrate_CarbonMangels': 0.3757416307926178,
 'CYP3A4_Veith': 0.08967520296573639,
 'Carcinogens_Lagunin': 0.277785062789917,
 'ClinTox': 0.11126358807086945,
 'DILI': 0.6482530832290649,
 'HIA_Hou': 0.9975790977478027,
 'NR-AR-LBD': 0.004215356428176165,
 'NR-AR': 0.010892867110669613,
 'NR-AhR': 0.006246307399123907,
 'NR-Aromatase': 0.00977207999676466,
 'NR-ER-LBD': 0.0029703795444220304,
 'NR-ER': 0.050187695771455765,
 'NR-PPAR-gamma': 0.0012441320577636361,
 'PAMPA_NCATS': 0.5115193128585815,
 'Pgp_Broccatelli': 0.013223608024418354,
 'SR-ARE': 0.06279595196247101,
 'SR-ATAD5': 0.002208638470619917,
 